# Basics of Text Generation with LLMs

<sup>This notebook is a part of the Natural Language Processing class at the University of Ljubljana, Faculty of Computer and Information Science.</sup>

---

Large Language Models generate text **one token at a time**, guided by a learned probability distribution over the vocabulary. The `pipeline` API exposes a rich set of generation parameters that shape how this process unfolds — controlling output length, randomness, quality, and diversity.

Everything in this notebook runs through a single object: the HuggingFace **`text-generation` pipeline**. We never load a model class or a tokenizer by hand, and we do not quantise — the model is loaded in its native precision.

We cover:

**Mechanics**
1. How LLMs generate text (autoregressive decoding)
2. Loading a model with `pipeline()` and reading what it returns
3. The defaults you inherit if you pass nothing

**Decoding parameters**

4. `max_new_tokens` — controlling output length
5. `do_sample` — greedy vs. stochastic decoding
6. `temperature` — sharpness of the probability distribution
7. `top_k` — vocabulary truncation to the top-*k* tokens
8. `top_p` — nucleus (dynamic vocabulary) sampling
9. `repetition_penalty` — penalising repeated tokens
10. `num_beams` — beam search
11. `no_repeat_ngram_size` — hard n-gram repetition penalty
12. `num_return_sequences` — generating multiple candidates

**Working with the pipeline**

13. Where generation stops — `min_new_tokens`, `stop_strings`
14. Steering the output format — system prompts and assistant prefill
15. Watching it generate — streaming
16. Reproducibility — seeds, and what they do *not* buy you

**Reasoning models**

17. Models that think before they answer — separating the thinking from the answer

## 1  Setup

We need the following libraries:

| Library | Purpose |
|---|---|
| `transformers` | The `pipeline` API and all generation parameters |
| `accelerate` | Device placement (`device_map="auto"`) and efficient loading |

No `bitsandbytes`: this notebook runs the model **unquantised**. A 1.5 B-parameter model in `bfloat16` needs roughly 3 GB of GPU memory, which fits comfortably on a free Colab T4.

In [ ]:
!pip install -q -U transformers accelerate

In [1]:
import warnings
from collections import Counter

warnings.filterwarnings("ignore")

from transformers import pipeline, set_seed, TextStreamer
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()

## 2  How LLMs Generate Text

Understanding the mechanics behind text generation makes every tuning parameter intuitive.

### Autoregressive Generation

An LLM is a function that maps a token sequence to a **probability distribution** over the next token:

$$P(w_t \mid w_1, w_2, \dots, w_{t-1})$$

Generation is **autoregressive**: the model picks one token, appends it to the input, and repeats — until a stop condition (`max_new_tokens` or an end-of-sequence token) is reached.

```
Token 1 → Token 2 → Token 3 → … → <EOS>
```

Because each step conditions on *all* previously generated tokens, early choices propagate forward through the entire sequence.

### The Probability Distribution

At each step the model produces a **logit vector** of size *V* (vocabulary size — typically 32 000–150 000 tokens). A softmax converts logits to probabilities:

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Sampling parameters (`temperature`, `top_k`, `top_p`) **reshape this distribution** before a token is drawn. Parameters such as `repetition_penalty` modify the logits directly, before the softmax.

### Why Sampling Parameters Matter

With a **peaked** distribution one token dominates — the model is deterministic but can get stuck in loops.
With a **flat** distribution many tokens are equally likely — outputs are diverse but often incoherent.

The sampling parameters let us tune this trade-off:

```
Low temperature / small k/p  →  conservative, focused, repetitive
High temperature / large k/p →  creative, diverse, potentially incoherent
```

## 3  Loading the Model with `pipeline()`

We use **Qwen/Qwen2.5-1.5B-Instruct** — a compact (1.5 B parameters), instruction-tuned model that runs comfortably on a single GPU.

`pipeline("text-generation", model=...)` does four things for us in one line:

1. downloads and instantiates the model,
2. downloads and instantiates the matching tokenizer,
3. places the weights on the available device (`device_map="auto"`),
4. wraps tokenise → `generate()` → decode into a single callable.

In [10]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

pipe = pipeline(
    task="text-generation",
    model=MODEL_NAME,
    dtype="auto",        # keep the checkpoint's native precision (bfloat16 for Qwen2.5)
    device_map="auto",   # put the weights on the GPU if there is one
)

print(f"Model:  {MODEL_NAME}")
print(f"Device: {pipe.model.device}")
print(f"Dtype:  {pipe.model.dtype}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model:  Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0
Dtype:  torch.bfloat16


### What the pipeline returns

An instruction-tuned model expects a **chat**: a list of `{"role", "content"}` dictionaries. The pipeline applies the model's chat template for us, so we never have to format special tokens by hand.

The return value is a list with one entry per generated sequence. In chat mode each entry contains the **whole conversation**, with the model's reply appended as the last message — which is why the helper below ends in `[-1]["content"]`.

In [11]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "Name one river in Slovenia. Answer with the name only."},
]

raw = pipe(messages, max_new_tokens=10, do_sample=False)

for turn in raw[0]["generated_text"]:
    print(f"{turn['role']:>10}: {turn['content']}")

    system: You are a helpful assistant.
      user: Name one river in Slovenia. Answer with the name only.
 assistant: Danube


### Two small helpers

Every example below calls one of these. They hide the message plumbing and pass any extra keyword argument straight through to the pipeline, which forwards it to `generate()`.

In [12]:
def generate(prompt, system="You are a helpful assistant.", **kwargs):
    """One completion. Extra keyword arguments are forwarded to generate()."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": prompt},
    ]
    result = pipe(messages, **kwargs)
    return result[0]["generated_text"][-1]["content"]


def generate_n(prompt, n, system="You are a helpful assistant.", **kwargs):
    """n completions of the same prompt, returned as a list of strings."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": prompt},
    ]
    result = pipe(messages, num_return_sequences=n, **kwargs)
    return [r["generated_text"][-1]["content"] for r in result]

### The defaults you inherit

If you call the pipeline without decoding arguments, you do **not** get neutral behaviour. You get whatever the model author shipped in the checkpoint's `generation_config.json`, filled in with the pipeline's own defaults where the checkpoint is silent.

For Qwen2.5-Instruct that means sampling is **on**, with a temperature, a `top_p`, a `top_k` and a repetition penalty already applied. Print them before you write "we used the default settings" in a methods section — those numbers *are* the default settings.

In [13]:
cfg = pipe.generation_config

for key in ["max_new_tokens", "do_sample", "temperature",
            "top_k", "top_p", "repetition_penalty", "num_beams",
            "no_repeat_ngram_size", "num_return_sequences"]:
    print(f"{key:>22} = {getattr(cfg, key, None)}")

        max_new_tokens = 256
             do_sample = True
           temperature = 0.7
                 top_k = 20
                 top_p = 0.8
    repetition_penalty = 1.1
             num_beams = 1
  no_repeat_ngram_size = 0
  num_return_sequences = 1


## 4  `max_new_tokens` — Controlling Output Length

`max_new_tokens` caps the **number of tokens generated in a single call**. Generation stops earlier if an end-of-sequence token appears first.

**Why it matters:** Too small and the response is truncated mid-thought; too large and generation is slow and may drift off-topic. Setting a value appropriate for your task is the cheapest optimisation available.

Note that truncation is *silent*: the pipeline returns a fragment without raising anything. If you parse the output automatically, a too-small `max_new_tokens` shows up as unexplained missing data rather than as an error.

In [14]:
PROMPT = "What does the attention mechanism in a transformer do? Answer in 2-3 sentences."

for n in [20, 80, 250]:
    print(f"=== max_new_tokens={n} ===")
    print(generate(PROMPT, do_sample=False, max_new_tokens=n))
    print()

=== max_new_tokens=20 ===
The attention mechanism in a transformer allows it to weigh different parts of its input sequence during the computation of

=== max_new_tokens=80 ===
The attention mechanism in a transformer allows it to weigh different parts of its input sequence during the computation of context vectors, focusing on relevant information for each position in the output sequence based on their importance. This enables transformers to handle long-range dependencies effectively and capture more complex patterns in the data compared to traditional RNNs or CNNs.

=== max_new_tokens=250 ===
The attention mechanism in a transformer allows it to weigh different parts of its input sequence during the computation of context vectors, focusing on relevant information for each position in the output sequence based on their importance. This enables transformers to handle long-range dependencies effectively and capture more complex patterns in the data compared to traditional RNNs or CNNs.

With `max_new_tokens=20` the explanation is cut off mid-sentence. At `80` we get a concise but almost complete answer. At `250` the model has room to elaborate with examples and details.

## 5  `do_sample` — Greedy vs. Stochastic Decoding

| Mode | Mechanism | Character |
|---|---|---|
| `do_sample=False` | Always pick the highest-probability token | Deterministic — identical output every run |
| `do_sample=True` | Draw from the probability distribution | Stochastic — different output each run |

**Greedy decoding** is reproducible and predictable — useful for factual or structured tasks.
**Sampling** introduces diversity — essential for creative, conversational, or data-augmentation tasks.

> Which one is the *default* depends on the checkpoint, not on the library — as section 3 showed, Qwen2.5-Instruct ships with `do_sample=True`. Pass the argument explicitly rather than relying on it.

In [16]:
FACTUAL_PROMPT = "What is the boiling point of water in Celsius, and why does pressure affect it?"

print("=== Greedy (do_sample=False) — run twice: output is identical ===")
print("RUN 1: ", generate(FACTUAL_PROMPT, do_sample=False, max_new_tokens=80))
print()
print("RUN 2: ", generate(FACTUAL_PROMPT, do_sample=False, max_new_tokens=80))

print("\n=== Sampling (do_sample=True) — run twice: output varies ===")
set_seed(1)
print("RUN 1: ", generate(FACTUAL_PROMPT, do_sample=True, max_new_tokens=80))
print()
set_seed(2)
print("RUN 2: ", generate(FACTUAL_PROMPT, do_sample=True, max_new_tokens=80))

=== Greedy (do_sample=False) — run twice: output is identical ===
RUN 1:  The boiling point of water at standard atmospheric pressure (1 atmosphere or 760 mmHg) is 100 degrees Celsius. However, if you increase the pressure above this value, the boiling point will also increase. This is because higher pressure means that there is more force pushing on the molecules of water, which makes them move slower and requires more energy to change their state from liquid

RUN 2:  The boiling point of water at standard atmospheric pressure (1 atmosphere or 760 mmHg) is 100 degrees Celsius. However, if you increase the pressure above this value, the boiling point will also increase. This is because higher pressure means that there is more force pushing on the molecules of water, which makes them move slower and requires more energy to change their state from liquid

=== Sampling (do_sample=True) — run twice: output varies ===
RUN 1:  The boiling point of water at standard atmospheric pressure (1 at

The two greedy outputs are word-for-word identical. The sampled outputs carry the same facts but differ in phrasing, sentence order, or which aspect they emphasise first.

## 6  `temperature` — Sharpness of the Distribution

Temperature *T* is applied by **dividing logits before the softmax**:

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

| Temperature | Effect | Output character |
|---|---|---|
| *T* < 1 | Distribution sharpened — the most likely token dominates | Conservative, repetitive |
| *T* = 1 | Distribution unchanged — model's native probabilities | Balanced |
| *T* > 1 | Distribution flattened — probabilities equalised | Creative, unpredictable |

> Setting `temperature → 0` approaches greedy decoding. Very high values (> 2) typically produce incoherent text.

Requires `do_sample=True` — temperature has no effect in greedy mode.

In [ ]:
CREATIVE_PROMPT = "Begin a short story about a robot who discovers an old library."

for temp in [0.1, 0.7, 1.5]:
    set_seed(42)
    print(f"=== temperature={temp} ===")
    print(generate(CREATIVE_PROMPT, do_sample=True, max_new_tokens=80, temperature=temp))
    print()

=== temperature=0.1 ===
In the quiet hum of his mechanical heart, the robot named Zephyr discovered an old library hidden behind a dilapidated wall in the outskirts of the city. The walls were covered in moss and vines, and the windows had long since been boarded up with rusted iron bars.

Zephyr's sensors picked up faint sounds coming from within the building - whispers, murmurs, and the distant

=== temperature=0.7 ===
In the quiet hum of a dimly lit room, a sleek, metallic figure stood before an ancient wooden door that creaked open with each attempt to push it closed. It was a peculiar sight - a robot, no more than a few feet tall, standing in what appeared to be a forgotten corner of an abandoned mansion.

The machine's eyes, glowing softly under the low light, scanned its

=== temperature=1.5 ===
In the vast emptiness of outer space, where the only company were the endless blackness of the stars and the humbling silence of the infinite cosmos, there lived an android named Zephyr.

At `temperature=0.1` the model plays it safe with generic phrasing. At `0.7` the prose has more personality. At `1.5` the language becomes vivid and surprising — though coherence may occasionally suffer.

### Seeing the distribution widen

Reading three stories tells you that temperature changes the *style*. To see what it does to the *distribution*, ask for a single token many times and count the answers.

Below we generate one token, 40 times, at three temperatures. The counts are a sample-based estimate of the next-token distribution the model is drawing from.

In [21]:
ONE_TOKEN_PROMPT = "Complete with a single word: The capital of France is famous for its"

for temp in [0.2, 1.0, 1.8]:
    set_seed(0)
    first_tokens = generate_n(ONE_TOKEN_PROMPT, n=40, do_sample=True,
                              temperature=temp, max_new_tokens=1)
    counts = Counter(t.strip() for t in first_tokens)
    print(f"temperature={temp:<4} {len(counts):>2} distinct continuations  {counts.most_common(6)}")

temperature=0.2   1 distinct continuations  [('E', 40)]
temperature=1.0   1 distinct continuations  [('E', 40)]
temperature=1.8   8 distinct continuations  [('E', 25), ('Not', 4), ('pal', 2), ('f', 2), ('Vers', 2), ('histor', 2)]


At a low temperature nearly every draw returns the same token: the distribution is peaked and sampling is effectively greedy. As the temperature rises the mass spreads out and rarer tokens start to win draws. This is the mechanism behind the stylistic differences above — and the reason a coding or classification task run at a high temperature will not reproduce.

## 7  `top_k` — Vocabulary Truncation

In **Top-*k* sampling** (Fan et al., 2018), only the *k* most probable tokens at each step are kept; the rest are set to zero probability before sampling.

```
Full distribution:    [0.30,  0.20,  0.15,  0.10,  0.08,  0.07, …]
top_k=3  →  keep:     [0.30,  0.20,  0.15,     0,     0,     0, …]  → re-normalise → sample
```

| `top_k` | Candidate pool | Output character |
|---|---|---|
| 1 | 1 token (greedy) | Deterministic |
| 10–20 | Narrow | Focused, conservative |
| 50–100 | Wide | Varied, creative |

In [22]:
for k in [1, 10, 50]:
    set_seed(42)
    print(f"=== top_k={k} ===")
    print(generate(CREATIVE_PROMPT, do_sample=True, max_new_tokens=80, top_k=k, temperature=0.9))
    print()

=== top_k=1 ===
In the quiet hum of his mechanical heart, the robot named Zephyr discovered an old library hidden behind a dilapidated wall in the outskirts of the city. The walls were covered in vines and moss, and the windows had long since been boarded up with rusted iron bars.

Zephyr's sensors picked up faint sounds coming from within the building, but he hesitated before venturing inside

=== top_k=10 ===
In the quiet hum of its mechanical gears, the robot had been programmed to explore and gather knowledge from every corner of the world. But one day, while traversing through a vast desert landscape, it stumbled upon an ancient-looking structure that seemed out of place amidst the barren sands.

Curiosity piqued, the robot approached the mysterious building with caution, scanning its surroundings for any signs of life or

=== top_k=50 ===
In the quiet hum of its mechanical engine, the robot had been programmed to patrol and maintain the city it resided in. But as it meandered thr

`top_k=1` produces the same greedy output every run. As *k* grows, sentence structure and vocabulary diversify — notice the difference in opening phrases and descriptive choices.

## 8  `top_p` — Nucleus Sampling

**Nucleus sampling** (Holtzman et al., 2020) selects the *smallest* set of tokens whose cumulative probability ≥ *p*, then samples from that set.

```
Sorted probabilities: [0.35, 0.25, 0.20, 0.10, 0.05, 0.03, 0.02, …]
top_p=0.80  →  keep until cumsum ≥ 0.80:  [0.35, 0.25, 0.20]  → re-normalise → sample
```

Unlike `top_k`, the **nucleus size adapts** at each step:
- Confident model (peaked distribution) → small nucleus (2–3 tokens)
- Uncertain model (flat distribution) → large nucleus (many tokens)

**Tip:** `top_p=0.9` with `top_k=0` is a popular nucleus-only configuration. The two filters can also be combined — the stricter one applies.

In [23]:
# Set top_k=0 to disable it and isolate the effect of top_p
for p in [0.5, 0.9, 1.0]:
    set_seed(42)
    print(f"=== top_p={p}  (top_k disabled) ===")
    print(generate(CREATIVE_PROMPT, do_sample=True, max_new_tokens=80,
                   top_p=p, top_k=0, temperature=0.9))
    print()

=== top_p=0.5  (top_k disabled) ===
In the quiet hum of its mechanical gears, the robot had been programmed to explore and gather information from various locations. One day, as it wandered through a vast industrial complex, it stumbled upon a small, abandoned building.

Curiosity piqued, the robot approached the door and pushed it open with its metal foot. Inside, it found itself in a cavernous space filled with towering bookshelves

=== top_p=0.9  (top_k disabled) ===
In the quiet hum of its mechanical machinery, the Alphamite continued its routine patrol around the starship Quasar Alpha One. Its sensors beeped steadily as it navigated through the ship's vast corridors, carefully scanning for any potential threats.

Suddenly, as it rounded a corner, the Alphamite caught a glimpse of something that made its circuits hum with excitement: an old library tucked

=== top_p=1.0  (top_k disabled) ===
In the quiet hum of its mechanical machinery, the Altered One turned in silent circles as i

At `top_p=0.5` the model is confined to its most confident choices, producing safe but predictable text. At `1.0` any token can be sampled, leading to more unusual — and occasionally less coherent — continuations.

## 9  `repetition_penalty` — Penalising Repeated Tokens

Before sampling, logits are **scaled down for any token that already appears** in the current sequence:

$$z'_i = \begin{cases} z_i / \text{penalty} & \text{if token } i \text{ has been generated before} \\ z_i & \text{otherwise} \end{cases}$$

| Value | Effect |
|---|---|
| `1.0` | No penalty — logits unchanged |
| `> 1.0` (e.g. 1.1–1.3) | Discourages repetition — typical range |
| `< 1.0` | Encourages repetition (rarely useful) |

This operates at the **individual-token level**: even a single prior occurrence reduces that token's probability. It is most noticeable in long or descriptive generations where the model tends to reuse favoured words.

**Caution for research use:** the penalty applies to *any* token already in the sequence, including tokens carried over from your prompt. On extraction or quotation tasks it can push the model away from the exact wording of the source text.

In [24]:
REPEAT_PROMPT = "Describe the ocean: its colour, its sounds, its movement, and its depth."

for penalty in [1.0, 1.2, 1.5]:
    set_seed(42)
    print(f"=== repetition_penalty={penalty} ===")
    print(generate(REPEAT_PROMPT, do_sample=True, max_new_tokens=100,
                   repetition_penalty=penalty, temperature=0.7, top_k=50))
    print()

=== repetition_penalty=1.0 ===
The ocean is a vast and complex system that spans thousands of miles across the globe. It is a living entity that is constantly changing and evolving. The ocean's colour, sounds, movement, and depth are all interconnected and shape the way we perceive it.

The ocean's colour is a blend of blues, greens, and yellows that vary depending on the time of day, the season, and the location. During the day, the ocean appears a deep blue due to the scattering of light by the

=== repetition_penalty=1.2 ===
The ocean is an immense body of water that covers over 71% of Earth's surface. It has various depths ranging from shallow estuaries to deep trenches with some parts being up to 36,000 feet (10,984 meters) below sea level.

Color:
The color of the ocean varies depending on several factors such as sunlight intensity, salinity levels, temperature variations at different layers in the water column, etc., but generally it appears blue-green or

=== repetition_penalty

At `1.0` the model may reuse the same descriptive words. At `1.2` repetitions drop noticeably. At `1.5` the model is forced to find alternative vocabulary at every step — inventive, but occasionally awkward.

## 10  `num_beams` — Beam Search

Instead of committing greedily to one token at each step, **beam search** keeps the *B* most probable partial sequences (beams) in parallel. At each step every beam is extended with all possible next tokens, and only the top-*B* combined sequences survive.

```
Step 1:  "The robot"  →  beam 1: "The robot carefully"  (log-prob −2.1)
                         beam 2: "The robot slowly"      (log-prob −2.4)
                         beam 3: "The robot gently"       (log-prob −2.6)
…each beam continues independently until EOS
```

| | Greedy (`num_beams=1`) | Beam (`num_beams=5`) |
|---|---|---|
| Speed | Fast | Slower (≈ ×B) |
| Memory | Baseline | ≈ ×B |
| Output | Locally optimal | Globally more probable |
| Diversity | Low | Low |

> Beam search excels at tasks with a **single best answer** (translation, summarisation). For open-ended generation, sampling usually sounds more natural.

Beam search requires `do_sample=False`.

In [25]:
SUMMARY_PROMPT = "Summarise in two sentences the importance of sleep for cognitive performance."

print("=== Greedy  (num_beams=1) ===")
print(generate(SUMMARY_PROMPT, do_sample=False, num_beams=1, max_new_tokens=100))

print("\n=== Beam search  (num_beams=5) ===")
print(generate(SUMMARY_PROMPT, do_sample=False, num_beams=5,
               max_new_tokens=100, early_stopping=True))

=== Greedy  (num_beams=1) ===
Sleep is essential for cognitive performance as it helps consolidate memories, improve attention and concentration, regulate emotions, and enhance decision-making abilities. Adequate sleep also plays a crucial role in maintaining overall brain health and preventing cognitive decline with age.

=== Beam search  (num_beams=5) ===
Sleep is essential for cognitive performance as it allows the brain to consolidate memories, regulate emotions, and maintain focus and attention. Without adequate sleep, cognitive functions such as memory, decision-making, and problem-solving are impaired, leading to decreased performance and increased risk of accidents and errors.


Both outputs are factually similar, but beam search tends to produce more fluent sentence structure because it can abandon locally attractive but globally sub-optimal word choices.

## 11  `no_repeat_ngram_size` — Hard N-gram Penalty

`no_repeat_ngram_size=n` sets the logit of any token that would **complete a previously-seen *n*-gram** to −∞, making it impossible to repeat any exact sequence of *n* consecutive tokens.

| Parameter | Mechanism | Granularity |
|---|---|---|
| `repetition_penalty` | Soft discount on any previously seen *individual* token | Single token |
| `no_repeat_ngram_size` | Hard ban on any repeated *n*-gram | Sequence of *n* tokens |

**Caution:** An aggressive setting (e.g. `n=2`) can prevent the model from repeating even natural phrases — for instance, a travel article about "New York" could only mention the city once.

In [26]:
for n in [0, 2, 4]:
    set_seed(42)
    label = "disabled" if n == 0 else f"ban any {n}-token repeat"
    print(f"=== no_repeat_ngram_size={n}  ({label}) ===")
    print(generate(REPEAT_PROMPT, do_sample=True, max_new_tokens=100,
                   no_repeat_ngram_size=n, temperature=0.7, top_k=50))
    print()

=== no_repeat_ngram_size=0  (disabled) ===
The ocean is an immense body of water that covers over 71% of the Earth's surface. It has a deep blue color due to the scattering of light by the water molecules in the upper layers. The deeper parts of the ocean have a darker shade of blue or even black due to the absence of sunlight.

As one dives deeper into the ocean, the pressure increases significantly. At depths of around 100 meters, the temperature begins to drop rapidly as the sun's rays cannot penetrate

=== no_repeat_ngram_size=2  (ban any 2-token repeat) ===
The ocean is an immense body of water that covers over 71% of the Earth's surface. It has a deep blue color due to the scattering of light by the water molecules in the upper layers. The deeper parts of it have more oxygen and sunlight, making them suitable for marine life.

The sound of waves crashing against the shore can be heard as you walk along the beach. As you go further out into the sea, the sound becomes softer and le

With `n=0` (disabled) short phrases may recur. At `n=2` no consecutive word pair appears twice, which sometimes forces awkward substitutions. At `n=4` natural language flows normally while still preventing structural repetition — a good default for longer outputs.

## 12  `num_return_sequences` — Several Candidates at Once

`num_return_sequences=n` returns *n* outputs from a **single** call. The prompt is encoded once and the *n* continuations are generated together, which is considerably cheaper than *n* separate calls.

Its meaning depends on the decoding strategy:

| Strategy | What the *n* sequences are | Constraint |
|---|---|---|
| `do_sample=True` | *n* independent draws from the distribution | — |
| `num_beams=B`, `do_sample=False` | The *n* highest-scoring beams | *n* ≤ *B* |
| `do_sample=False`, `num_beams=1` | Not meaningful — greedy has one answer | *n* must be 1 |

With the pipeline the *n* candidates come back as *n* entries in the returned list, which is what `generate_n()` unpacks.

In [27]:
IDEA_PROMPT = "Suggest a title for a survey on how families experience energy price rises. Title only."

print("=== 4 independent samples (do_sample=True, temperature=1.0) ===")
for i, s in enumerate(generate_n(IDEA_PROMPT, n=4, do_sample=True,
                                 temperature=1.0, max_new_tokens=25), 1):
    print(f"  {i}. {s}")

print("\n=== top 4 of 6 beams (do_sample=False, num_beams=6) ===")
for i, s in enumerate(generate_n(IDEA_PROMPT, n=4, do_sample=False,
                                 num_beams=6, max_new_tokens=25), 1):
    print(f"  {i}. {s}")

=== 4 independent samples (do_sample=True, temperature=1.0) ===
  1. "Family Energy Budgets: Understanding the Impact of Price Rises"
  2. "Survey: How Families Are Coping with Energy Price Rises"
  3. "Family Perspectives: Exploring the Impact of Energy Price Rises on Daily Life and Budgeting"
  4. "Energy Prices Upset: How Families Navigate Financial Challenges During Price Rises"

=== top 4 of 6 beams (do_sample=False, num_beams=6) ===
  1. "Energy Price Rises: Families' Experiences and Coping Strategies"
  2. "Energy Price Rises: How Families Are Adapting and Coping"
  3. "Energy Price Rises: Families' Experiences and Impact on Daily Life"
  4. "Energy Price Rises: Families' Experiences and Adaptations"


### Why this matters beyond saving time

Repeated sampling turns the model's instability into a **measurement**. Ask the same closed question *n* times and the share of answers that agree is a cheap confidence signal: near-unanimous answers are usually safe, split answers mark the cases a human should read.

This is the idea behind *self-consistency* (Wang et al., 2023), and on a coding task it is often more informative than anything the model says about its own certainty.

In [32]:
CLASSIFY = (
    "Coding scheme: support, oppose, unclear.\n\n"
    "Public comment: {}\n\n"
    "Assign exactly one label from the scheme. Answer with the label only."
)

COMMENTS = [
    "I fully support the project. We need the energy and the jobs it would bring.",
    "I am not against wind power in principle, but not on that particular ridge.",
]

for comment in COMMENTS:
    set_seed(9)
    answers = generate_n(CLASSIFY.format(comment), n=10, do_sample=True,
                         temperature=0.8, max_new_tokens=5)
    votes = Counter()
    for a in answers:
        words = a.strip().lower().strip(".\"'").split()
        if words:
            votes[words[0]] += 1

    label, count = votes.most_common(1)[0]
    print(f"{count / sum(votes.values()):.0%} agreement -> {label:<8} {dict(votes)}")
    print(f"    {comment}\n")

90% agreement -> support  {'support': 9, 'supporting': 1}
    I fully support the project. We need the energy and the jobs it would bring.

50% agreement -> oppose   {'oppose': 5, 'support': 5}
    I am not against wind power in principle, but not on that particular ridge.



The unambiguous comment should come back unanimous. The conditional one ("not on that particular ridge") is where the votes split — and that split is the signal telling you which cases to read yourself.

## 13  Where Generation Stops

Three mechanisms end a generation, and they compete: whichever fires first wins.

| Parameter | Effect |
|---|---|
| `max_new_tokens` | Hard upper bound on generated tokens |
| the model's EOS token | Natural end of turn — why short answers finish early |
| `min_new_tokens` | Floor: EOS is forbidden until this many tokens exist |
| `stop_strings` | Stop as soon as one of these strings appears in the output |

`stop_strings` is checked against the *decoded text*, so it can match anything — `"\n"`, `"###"`, `"</answer>"`. Because that check needs a tokenizer, `generate()` insists on being handed one; through the pipeline that is `tokenizer=pipe.tokenizer`.

`min_new_tokens` is the mirror image of `max_new_tokens`, and is mostly useful when a model answers a substantive question with a single word.

In [33]:
LIST_PROMPT = "List three methods of qualitative data collection, one per line."

print("=== no stop string ===")
print(generate(LIST_PROMPT, do_sample=False, max_new_tokens=60))

print("\n=== stop_strings=['\\n'] — cut at the first line break ===")
print(generate(LIST_PROMPT, do_sample=False, max_new_tokens=60,
               stop_strings=["\n"], tokenizer=pipe.tokenizer))

print("\n=== min_new_tokens=40 — EOS suppressed until 40 tokens exist ===")
print(generate("Answer in one word: is qualitative coding subjective?",
               do_sample=False, max_new_tokens=60, min_new_tokens=40))

=== no stop string ===
1. Interviews
2. Focus groups
3. Observations

=== stop_strings=['\n'] — cut at the first line break ===
1. Interviews


=== min_new_tokens=40 — EOS suppressed until 40 tokens exist ===
Yes. Qualitative coding can be subjective because it involves interpreting and analyzing data to identify patterns, themes, or insights that may not be immediately obvious. The interpretation of the data often depends on the researcher's understanding, experience, and biases.


The stop string keeps only the first item — useful when you want one field per call and do not intend to parse the rest. The `min_new_tokens` example shows the opposite failure: forced past its natural stopping point, the model keeps talking, and the extra tokens are padding rather than information.

## 14  Steering the Output Format

Decoding parameters control *how* tokens are chosen. They cannot make the model answer in the shape you need — that is the job of the prompt, and of two pipeline features.

**The system message** sets the persona and the rules for the whole conversation. Our helper exposes it as `system=`.

**Assistant prefill** is the stronger tool: end the chat with an `assistant` message and the pipeline continues it instead of starting a new one. The model cannot preface its answer with "Sure! Here is…" because it is already mid-sentence. This is how you reliably get JSON, a bare label, or a table.

In [34]:
COMMENT = "If the access road is rebuilt and noise limits are respected, I would back it."

print("=== plain instruction ===")
print(generate(f"Return JSON with keys 'label' and 'confidence' for this comment: {COMMENT}",
               do_sample=False, max_new_tokens=60))

print("\n=== strict system message ===")
print(generate(f"Classify this comment: {COMMENT}",
               system="You are a text coder. Reply with one word: support, oppose or unclear. Never explain.",
               do_sample=False, max_new_tokens=10))

print("\n=== assistant prefill: the reply is forced to start inside the JSON ===")
prefilled = [
    {"role": "system",    "content": "You label public comments."},
    {"role": "user",      "content": f"Return JSON with keys 'label' and 'confidence' for: {COMMENT}"},
    {"role": "assistant", "content": '{"label": "'},
]
print(pipe(prefilled, do_sample=False, max_new_tokens=30)[0]["generated_text"][-1]["content"])

=== plain instruction ===
```json
{
  "label": "Support",
  "confidence": 0.95
}
```

=== strict system message ===
support

=== assistant prefill: the reply is forced to start inside the JSON ===
{"label": "Support", "confidence": 0.95}


The plain instruction usually works, but may wrap the JSON in prose or a code fence. The prefill cannot: the assistant turn already contains `{"label": "`, so the first token the model is allowed to produce is the label itself, and the output parses without cleanup.

## 15  Watching It Generate — Streaming

By default the pipeline returns only once generation has finished, which feels slow for long outputs. A **streamer** prints each token as it is produced. Nothing about the result changes — it is purely a matter of when you see it.

`TextStreamer` writes to stdout; `TextIteratorStreamer` yields tokens to your own code and is what you would use behind a web interface. Streaming handles one sequence at a time, so do not combine it with batching or `num_return_sequences`.

In [35]:
streamer = TextStreamer(pipe.tokenizer, skip_prompt=True, skip_special_tokens=True)

_ = generate("Explain in three sentences why survey response rates have declined since 2000.",
             do_sample=False, max_new_tokens=120, streamer=streamer)

Survey response rates have declined since 2000 due to several factors, including the rise of online surveys and the increasing use of mobile devices for communication, which can make it difficult for respondents to complete surveys quickly or easily. Additionally, changes in consumer behavior and preferences may also contribute to lower response rates, as people may be less willing to participate in surveys that take time away from their daily activities or require them to provide sensitive information. Finally, concerns about data privacy and security may also deter some individuals from participating in surveys, particularly if they perceive that their responses will be used in ways that they


## 16  Reproducibility

Sampling draws from a pseudo-random generator, so `set_seed(42)` before a call makes that call repeatable. That is worth doing — and it is worth being precise about what it buys you.

| Property | Delivered by a seed? |
|---|---|
| **Reproducible** — same code, same machine, same output | Yes |
| **Stable** — the answer would not have changed under a different seed | No |
| **Correct** — the answer is right | No |

A fixed seed freezes one draw out of the distribution. If the model is genuinely uncertain, the seed hides that uncertainty rather than removing it. The honest order of operations is: measure the variability first (section 12), then fix the seed for the run you publish.

Two further caveats. Greedy decoding is deterministic without any seed, but results can still differ across GPU models, library versions, and even batch sizes, because floating-point reductions are not associative. And when you report your method, the seed alone is not enough: the model name and revision, the dtype, and every decoding parameter belong in the methods section too.

In [ ]:
print("same seed, twice:")
for _ in range(2):
    set_seed(123)
    print("  ", generate(IDEA_PROMPT, do_sample=True, temperature=1.0, max_new_tokens=20))

print("\ndifferent seeds — the instability the seed was hiding:")
for seed in [1, 2, 3]:
    set_seed(seed)
    print(f"  seed={seed}: {generate(IDEA_PROMPT, do_sample=True, temperature=1.0, max_new_tokens=20)}")

## 17  Reasoning Models

Everything so far used an ordinary instruction model: it starts answering straight away. **Reasoning models** (DeepSeek-R1, Qwen3, OpenAI's o-series, the "thinking" modes of Claude and Gemini) are trained to first write out a chain of thought, and only then answer. In open models the reasoning comes back as plain text between `<think>` and `</think>`, in the same reply as the answer:

```
<think>
The commenter would back the project, but only if two conditions are met ...
</think>

unclear
```

Nothing about generation has changed. It is still the token-by-token process of section 2: the thinking is simply more generated tokens, and every decoding parameter in this notebook applies to it. For research use that has three consequences:

1. **You have to split the reply.** A parser that reads the first word of the reply (as in section 12) reads `<think>`. Keep the answer for analysis, and store the thinking separately.
2. **`max_new_tokens` has to cover both.** Thinking often runs to hundreds or thousands of tokens. If generation stops inside the thinking, there is no answer at all: the truncation problem of section 4, only worse.
3. **The thinking is not a record of how the model decided.** It is generated text, and studies find it often leaves out what actually drove the answer. Treat it as a rationale the model wrote, not as an explanation you can cite.

We use **Qwen3-1.7B**, the next generation of the model family used above. It is about 3.4 GB in `bfloat16`, so it fits on a T4 next to the first model, and it has a switch that turns thinking on or off.

In [36]:
REASONING_MODEL = "Qwen/Qwen3-1.7B"

reasoner = pipeline(
    "text-generation",
    model=REASONING_MODEL,
    dtype="auto",
    device_map="auto",
)

print(f"{REASONING_MODEL} on {reasoner.model.device} ({reasoner.model.dtype})")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Qwen/Qwen3-1.7B on cuda:0 (torch.bfloat16)


### One reply, two parts

We ask it to code the conditional comment from section 14. Qwen recommends sampling for thinking mode (`temperature=0.6`, `top_p=0.95`, `top_k=20`). Greedy decoding tends to make reasoning models loop, repeating the same thought until the token budget runs out.

In [37]:
THINKING_DECODING = dict(do_sample=True, temperature=0.6, top_p=0.95, top_k=20)

question = [{
    "role": "user",
    "content": "Classify this public comment about a proposed wind farm as support, oppose or unclear. "
               f"Give the label only.\n\nComment: {COMMENT}",
}]

raw = reasoner(question, max_new_tokens=1024, **THINKING_DECODING)[0]["generated_text"][-1]["content"]
print(raw)

<think>
Okay, let's see. The user wants me to classify this public comment as support, oppose, or unclear. The comment says, "If the access road is rebuilt and noise limits are respected, I would back it." 

First, I need to understand the stance of the comment. The person is saying they would support the wind farm if two conditions are met: the access road is rebuilt and noise limits are respected. So, they're not against the project unless those specific conditions aren't met. 

Now, the question is whether this is support, oppose, or unclear. Since they mention backing the project under certain conditions, it's not outright oppose. They're not opposing the project in general, but they're conditional on the access road and noise limits. 

But wait, the comment doesn't mention any opposition to the project. It's just saying they would support it if those two things are done. So, the comment is in favor of the project under specific circumstances. That would be support. 

But maybe the

The reply is one string: the thinking, then `</think>`, then the answer. `split_thinking()` separates them, and handles the two cases that break a naive split:

- **No closing tag.** Generation stopped in the middle of the thinking. There is no answer, and the function returns `None` so that the case shows up as missing rather than being parsed as a label.
- **No opening tag.** Some chat templates put `<think>` into the *prompt*, so the reply starts mid-thought and only the closing tag appears.

It splits at the *last* `</think>`, as Qwen's own examples do, in case the model mentions the tag inside its reasoning.

In [38]:
def split_thinking(reply):
    """Split a reasoning model's reply into (thinking, answer).

    answer is None if the reply stopped before the thinking was finished.
    """
    if "</think>" not in reply:
        if "<think>" in reply:                        # opened but never closed: cut off mid-thought
            return reply.split("<think>", 1)[1].strip(), None
        return "", reply.strip()                      # the model did not think at all
    thinking, answer = reply.rsplit("</think>", 1)
    thinking = thinking.replace("<think>", "", 1)     # may be missing if the template put it in the prompt
    return thinking.strip(), answer.strip()


def n_tokens(text):
    return len(reasoner.tokenizer.encode(text))


thinking, answer = split_thinking(raw)
print(f"THINKING ({n_tokens(thinking)} tokens):\n{thinking[:800]}{' ...' if len(thinking) > 800 else ''}\n")
print(f"ANSWER ({n_tokens(answer)} tokens): {answer!r}")

THINKING (329 tokens):
Okay, let's see. The user wants me to classify this public comment as support, oppose, or unclear. The comment says, "If the access road is rebuilt and noise limits are respected, I would back it." 

First, I need to understand the stance of the comment. The person is saying they would support the wind farm if two conditions are met: the access road is rebuilt and noise limits are respected. So, they're not against the project unless those specific conditions aren't met. 

Now, the question is whether this is support, oppose, or unclear. Since they mention backing the project under certain conditions, it's not outright oppose. They're not opposing the project in general, but they're conditional on the access road and noise limits. 

But wait, the comment doesn't mention any opposition to t ...

ANSWER (1 tokens): 'support'


### Thinking on and off: what does it cost?

Qwen3's chat template has an `enable_thinking` switch, which the pipeline passes on through `tokenizer_encode_kwargs`. With thinking off, the template inserts an empty `<think></think>` block into the prompt, and the model answers directly.

The switch is specific to Qwen3. Other reasoning models do it differently: DeepSeek-R1 always thinks, and some models read an instruction such as "Reasoning: low" from the system prompt. The model card says which.

In [39]:
import time

for thinking_on in [True, False]:
    start = time.time()
    reply = reasoner(
        question,
        max_new_tokens=1024,
        tokenizer_encode_kwargs={"enable_thinking": thinking_on},
        **THINKING_DECODING,
    )[0]["generated_text"][-1]["content"]
    thinking, answer = split_thinking(reply)
    print(f"thinking={thinking_on!s:<5}  thinking tokens={n_tokens(thinking):4d}  "
          f"time={time.time() - start:5.1f}s  answer={answer!r}")

thinking=True   thinking tokens= 480  time= 20.8s  answer='support'
thinking=False  thinking tokens=   0  time=  0.1s  answer='Support'


For a one-word label, thinking usually costs tens of times more tokens and time, and the label is often the same. It pays off where there are genuine steps (arithmetic, conditional rules, long instructions), which is the scope of chain-of-thought in the *Prompting* notebook. Because the thinking is sampled, run the cell a few times before concluding anything. Reasoning is not exempt from section 12.

### When the budget runs out

The same question with a token budget that would be generous for an ordinary model:

In [40]:
reply = reasoner(question, max_new_tokens=60, **THINKING_DECODING)[0]["generated_text"][-1]["content"]
thinking, answer = split_thinking(reply)

print(f"thinking so far: {thinking[-200:]!r}")
print(f"answer: {answer!r}")      # None: the budget ran out while the model was still thinking

thinking so far: 'the public comment as support, oppose, or unclear. The comment says, "If the access road is rebuilt and noise limits are respected, I would back it." \n\nFirst, I need to understand the stance here. The'
answer: None


Sixty tokens are spent before the model gets to the answer, and `answer` is `None`. Across a corpus this is missing data that is **not** missing at random: the model thinks longest about the ambiguous cases, so those are the ones that lose their label. Count the `None`s, and raise the budget until there are none.

**For research use:**

- Store three things per case: the raw reply, the thinking and the answer. Analyse only the answer.
- Never parse a label out of the thinking. It often mentions every label while weighing them.
- Report whether thinking was on, the token budget, and the decoding parameters, just as for any other model.
- Treat the length of the thinking as a possible difficulty signal, like the agreement rate in section 12, but check that it tracks the cases you find hard before you rely on it.

## Summary

**Decoding parameters**

| Parameter | What it controls | Practical default |
|---|---|---|
| `max_new_tokens` | Maximum output length | Task-dependent (50–500) |
| `min_new_tokens` | Minimum output length | Unset |
| `do_sample` | Greedy vs. stochastic | `True` for creative, `False` for factual |
| `temperature` | Distribution sharpness | 0.6–0.9 for most tasks |
| `top_k` | Hard vocabulary size limit | 40–50 |
| `top_p` | Dynamic vocabulary size limit | 0.9–0.95 |
| `repetition_penalty` | Soft per-token repetition cost | 1.1–1.2 |
| `num_beams` | Beam search width | 1 (sampling) or 4–5 (beam) |
| `no_repeat_ngram_size` | Hard n-gram repetition ban | 3–4 for long-form text |
| `num_return_sequences` | Number of outputs per call | 1, or 5–20 for agreement rates |
| `stop_strings` | Extra textual stop condition | Unset (needs `tokenizer=`) |

**Pipeline arguments**

| Argument | Where it goes | Purpose |
|---|---|---|
| `dtype`, `device_map` | `pipeline(...)` | Precision and device placement |
| `batch_size` | the call | Parallel generation over a list of inputs |
| `streamer` | the call | Token-by-token output |
| `system=` / assistant prefill | the chat | Output format and persona |
| `tokenizer_encode_kwargs` | the call | Chat-template options, e.g. `{"enable_thinking": False}` for Qwen3 |

---

**Recipe for open-ended / creative generation:**
```python
generate(prompt, do_sample=True, temperature=0.8, top_p=0.9,
         repetition_penalty=1.1, max_new_tokens=200)
```

**Recipe for factual / structured output:**
```python
generate(prompt, do_sample=False, num_beams=4,
         no_repeat_ngram_size=3, early_stopping=True, max_new_tokens=150)
```

**Recipe for coding a corpus, with an uncertainty estimate:**
```python
labels = generate_n(prompt, n=10, do_sample=True, temperature=0.7, max_new_tokens=5)
winner, votes = Counter(labels).most_common(1)[0]   # agreement = votes / len(labels)
```

**Recipe for a reasoning model:**
```python
reply = reasoner(messages, max_new_tokens=2048, do_sample=True, temperature=0.6,
                 top_p=0.95, top_k=20)[0]["generated_text"][-1]["content"]
thinking, answer = split_thinking(reply)            # answer is None if the budget ran out
```

## References

* HuggingFace `pipeline` documentation — https://huggingface.co/docs/transformers/main_classes/pipelines
* Text generation parameters (`GenerationConfig`) — https://huggingface.co/docs/transformers/main_classes/text_generation
* Generation strategies — https://huggingface.co/docs/transformers/generation_strategies
* Chat templates — https://huggingface.co/docs/transformers/chat_templating
* How to generate text (HuggingFace blog) — https://huggingface.co/blog/how-to-generate
* Top-K sampling — Fan et al. (2018) — https://arxiv.org/abs/1805.04833
* Nucleus / Top-p sampling — Holtzman et al. (2020) — https://arxiv.org/abs/1904.09751
* Self-consistency — Wang et al. (2023) — https://arxiv.org/abs/2203.11171
* Qwen2.5 model — https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
* Qwen3 model (thinking and non-thinking modes) — https://huggingface.co/Qwen/Qwen3-1.7B
* Qwen3 Technical Report — Yang et al. (2025) — https://arxiv.org/abs/2505.09388
* DeepSeek-R1: incentivising reasoning in LLMs via reinforcement learning — DeepSeek-AI (2025) — https://arxiv.org/abs/2501.12948
* Reasoning models don't always say what they think — Chen et al. (2025) — https://arxiv.org/abs/2505.05410